# Topic Modelling Starter

This notebook sets up a reusable workflow for LDA and BERTopic on the news dataset.

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer



In [2]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
DATA_PATH = r"C:\Users\WEWL\OneDrive - Capco\Desktop\ADS Assessment\Data\NLP_data.csv"

TEXT_COLUMN = "content"
SAMPLE_SIZE = 5000

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COLUMN]).copy()
df = df[["article_id", "title", "category", TEXT_COLUMN]].copy()

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df.head()

,article_id,title,category,content
0,63642,Young teen wins top science prize for soap tha...,Ethiopia,"A 14-year-old boy has been named ""America's to..."
1,13370,"Vehicular homicide suspect who ""reeked of alco...",United States,"Ting Ye, 26, ""reeked of alcohol"" when she was ..."
2,69158,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,Kelly Nam plates a dessert at New York City’s ...
3,97723,"Flula Borg hilarious deconstructs that weird ""...",History,"Ok, one last Halloween post and I'll abandon t..."
4,84216,Korat zoo welcomes new member - sun bear,Myanmar,Nakhon Ratchasima Zoo has a new member - a one...


In [8]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if len(token) > 2 and token not in stop_words]
    return tokens

df["tokens"] = df[TEXT_COLUMN].apply(clean_text)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

df[["title", "category", "clean_text"]].head()

,title,category,clean_text
0,Young teen wins top science prize for soap tha...,Ethiopia,year old boy named america top young scientist...
1,"Vehicular homicide suspect who ""reeked of alco...",United States,ting reeked alcohol pulled driver seat mph rol...
2,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,kelly nam plates dessert new york city joomak ...
3,"Flula Borg hilarious deconstructs that weird ""...",History,one last halloween post abandon topic next yea...
4,Korat zoo welcomes new member - sun bear,Myanmar,nakhon ratchasima zoo new member one month old...


In [9]:
print(df.shape)
df["category"].fillna("Unknown").value_counts().head(10)

(5000, 6)


category
Stock          180
Technology     120
Finance        116
Real estate    115
Health         109
Canada         107
Education       97
News            93
Food            90
Jobs            86
Name: count, dtype: int64

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.90,
    min_df=20,
    ngram_range=(1, 2)
)

doc_term_matrix = vectorizer.fit_transform(df["clean_text"])

In [11]:
for n in [5, 8, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=n,
        random_state=42,
        learning_method="batch"
    )
    lda_model.fit(doc_term_matrix)
    print(f"\nTop words for {n} topics:")
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")


Top words for 5 topics:
Topic 1: nov, announced, new, global, globe, newswire, president, globe newswire, united, nov globe
Topic 2: world, news, new, cup, world cup, south, watch, live, zealand, new zealand
Topic 3: new, getty, years, images, like, getty images, time, week, life, day
Topic 4: report, free, free report, quarter, recent, according, company, nyse, shares, second
Topic 5: year, thursday, said, million, israel, gaza, reported, amazon, people, hamas

Top words for 8 topics:
Topic 1: nov, announced, globe, newswire, globe newswire, nov globe, global, series, states, today
Topic 2: world, news, cup, world cup, new, watch, south, zealand, new zealand, live
Topic 3: years, new, time, life, youtube, long, car, amp, good, way
Topic 4: report, free, free report, recent, according, quarter, company, nyse, shares, quarter according
Topic 5: year, thursday, million, apple, new, reported, market, announced, business, bitcoin
Topic 6: new, like, york, new york, city, people, home, med

In [12]:
def print_lda_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

feature_names = vectorizer.get_feature_names_out()
print_lda_topics(lda_model, feature_names)

Topic 1: season, food, start, new, friday, international, october, countries, event, announced
Topic 2: news, watch, live, sports, panama, apple, cbs, copyright, north, exclusive
Topic 3: years, life, time, games, google, youtube, day, new, week, car
Topic 4: report, free, free report, recent, quarter, according, company, shares, nyse, quarter according
Topic 5: thursday, year, reported, new, announced, people, bitcoin, data, company, quarter
Topic 6: new, november, amp, like, media, home, social, city, led, instagram
Topic 7: million, year, business, chief, space, net, big, health, financial, president
Topic 8: getty, images, israel, getty images, said, gaza, state, hamas, china, war
Topic 9: world, new, cup, world cup, zealand, new zealand, intelligence, artificial, artificial intelligence, press
Topic 10: new, year, old, said, sign, story, thursday, year old, music, news
Topic 11: nov, global, new, globe, newswire, globe newswire, nov globe, york, new york, market
Topic 12: united, 

In [13]:
lda_topic_matrix = lda_model.transform(doc_term_matrix)
df["lda_topic"] = lda_topic_matrix.argmax(axis=1)
df[["title", "category", "lda_topic"]].head(10)

,title,category,lda_topic
0,ãBlenderãã¦ã¼ã¶ã¼ã®ãç¥­ããBlende...,YouTube,8
1,Morning news brief,News,7
2,A colorful slice of city life: Winners of the ...,Photography,7
3,"Hanwha Ocean eyes submarine exports to Canada,...",Philippines,7
4,Banco Santander-Chile (NYSE:BSAC) Upgraded by ...,America,3
5,ç¾å½åäº¿æ æ¥ç»è´¹é­è´¨ç(å¾),COVID,7
6,"More foreign citizens, including about 400 Ame...",Food,7
7,Uzodimma accuses Ajaero of meddling in Imo pol...,Politics,7
8,Armenia's Pashinyan hopes peace deal with Azer...,Armenia,9
9,Tech Billionaires' Quest to Build a New City i...,Real estate,4


In [16]:
coherence_scores = {}
for n in [5, 8, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=n,
        random_state=42,
        learning_method="batch"
    )
    lda_model.fit(doc_term_matrix)
    
    topic_terms = []
    for topic in lda_model.components_:
        top_indices = topic.argsort()[:-11:-1]
        topic_terms.append([feature_names[i] for i in top_indices])
    
    coherence_model = CoherenceModel(
        topics=topic_terms,
        texts=df["tokens"],
        dictionary=dictionary,
        coherence="c_v"
    )
    coherence_scores[n] = coherence_model.get_coherence()

print("Coherence scores for different k:")
for k, score in coherence_scores.items():
    print(f"k={k}: {score:.4f}")

Coherence scores for different k:
k=5: 0.6030
k=8: 0.4735
k=10: 0.4772
k=12: 0.4251


In [11]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
bertopic_topics, bertopic_probs = topic_model.fit_transform(df["clean_text"].tolist())
df["bertopic_topic"] = bertopic_topics

2026-03-27 14:15:55,915 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-03-27 14:16:16,025 - BERTopic - Embedding - Completed ✓
2026-03-27 14:16:16,026 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-27 14:16:33,041 - BERTopic - Dimensionality - Completed ✓
2026-03-27 14:16:33,043 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-27 14:16:33,947 - BERTopic - Cluster - Completed ✓
2026-03-27 14:16:33,952 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-27 14:16:34,057 - BERTopic - Representation - Completed ✓


In [12]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1586,-1_chars_new_one_year,"[chars, new, one, year, world, getty, time, sa...",[space part future inc international media gro...
1,0,489,0_quarter_recent_according_filing,"[quarter, recent, according, filing, inc, seco...",[capital wealth advisors llc acquired new posi...
2,1,300,1_israel_gaza_hamas_israeli,"[israel, gaza, hamas, israeli, war, iran, pale...",[united nations china iran multitude arab nati...
3,2,152,2_pro_amazon_black_apple,"[pro, amazon, black, apple, samsung, iphone, d...",[tiktok legends gio iphone pro max iphone pro ...
4,3,115,3_film_series_films_comedy,"[film, series, films, comedy, show, movie, doc...",[ranbir also spoke film tried find similarity ...
5,4,95,4_crypto_bitcoin_cryptocurrency_reserve,"[crypto, bitcoin, cryptocurrency, reserve, fed...",[bitcoin cryptocurrencies advanced thursday li...
6,5,94,5_market_newswire_globe_nov,"[market, newswire, globe, nov, size, global, u...",[dublin nov globe newswire global power steeri...
7,6,75,6_ukraine_russia_russian_ukrainian,"[ukraine, russia, russian, ukrainian, moscow, ...",[president ukraine october president ukraine v...
8,7,70,7_cup_world_cricket_india,"[cup, world, cricket, india, england, australi...",[icc men cricket world cup became attended icc...
9,8,63,8_climate_storm_change_tropical,"[climate, storm, change, tropical, florida, hu...",[post tropical storm philippe expected make la...


In [13]:
topic_model.get_topic(0)

[('quarter', np.float64(0.04902388896291638)),
 ('recent', np.float64(0.047086933702961374)),
 ('according', np.float64(0.04471758943067745)),
 ('filing', np.float64(0.04248474001282883)),
 ('inc', np.float64(0.03857626273875938)),
 ('second', np.float64(0.03619182982048854)),
 ('free', np.float64(0.03524078263698651)),
 ('nyse', np.float64(0.03458774507095284)),
 ('llc', np.float64(0.03344450667203876)),
 ('holdings', np.float64(0.03265728776776795))]

In [14]:
# Run in Jupyter to open the interactive visualisation.\n
topic_model.visualize_barchart(top_n_topics=10)

In [15]:
topic_model.visualize_topics()

In [16]:
topic_model_17 = BERTopic(nr_topics=17)
topics_17, probs_17 = topic_model_17.fit_transform(df["clean_text"].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
